# 03. RetinaNet (одностадійний, з anchors)

Протокол той самий, що в `02_faster_rcnn`: дані, ResNet50-FPN з вагами COCO, 6 епох, SGD, batch 8, AMP, оцінка в оригінальних координатах.
Відмінності: lr 0.005 (так радять для RetinaNet при batch 8), детекція з рівнів P3–P7 (stride 8–128), стадії пропозицій немає.
У базовій моделі 9 anchors на позицію (3 масштаби × 3 співвідношення), focal loss (α=0.25, γ=2), позитив при IoU ≥ 0.5, негатив при IoU < 0.4, регресія L1.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ostapmn/ucu_ai_hw_2.git"
if Path("/kaggle/working").exists() and not Path("common.py").exists():
    repo = Path("/kaggle/working/repo")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
    os.chdir(repo / "code")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, ".")
DEBUG = os.environ.get("HW2_DEBUG") == "1"

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch

import analysis as an
import common as c
import tv_models as tv

c.set_seed()
P = c.paths()
OUT = P.out / "one_stage_anchor"
print("відновлено з Input:", c.restore_from_inputs(P, "one_stage_anchor"))
GT = P.prep / "val" / "annotations_eval.json"
RAW = P.data / "val" / "annotations_coco.json"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ANCHORS = tv.anchors_from_kmeans(P.prep / "anchors_960.json")
PROTOCOL = dict(epochs=6, batch=8, lr=0.005, milestones=(4, 5), warmup=300, workers=4)
if DEBUG:
    PROTOCOL.update(epochs=1, batch=2, warmup=5, workers=0, milestones=(1,))
print("anchors з k-means (база на рівнях P3-P7):", ANCHORS)

## 1. Скільки об'єктів покривають anchors (до навчання)

Для кожного GT-об'єкта з вибірки train беремо найкращий IoU серед усіх anchors на реальній сітці моделі. Якщо жоден anchor не перетинає об'єкт вище порогу позитиву, під час навчання цей об'єкт не отримає жодного позитивного прикладу.

In [ ]:
CONFIGS = {
    "стандартні anchors (32-512), IoU ≥ 0.5": (dict(), 0.5),
    "k-means anchors, IoU ≥ 0.5": (dict(anchor_base=ANCHORS), 0.5),
    "k-means anchors, IoU ≥ 0.4": (dict(anchor_base=ANCHORS), 0.4),
}
cover = {}
for label, (cfg, thr) in CONFIGS.items():
    m = tv.build_model("retinanet", pretrained=False, **cfg)
    cover[label] = tv.anchor_match_stats(m, P.prep, "train", every=200 if DEBUG else 50, fg_iou=thr)
cover = pd.DataFrame(cover).T
c.save_json(cover.reset_index().to_dict("records"), OUT / "anchor_coverage.json")
cover.round(3)

## 2. Експерименти

In [ ]:
EXPERIMENTS = {
    "retina_base": dict(),
    "retina_anchors": dict(anchor_base=ANCHORS),
    "retina_anchors_iou": dict(anchor_base=ANCHORS, fg_iou_thresh=0.4, bg_iou_thresh=0.3),
    "retina_anchors_iou_giou": dict(anchor_base=ANCHORS, fg_iou_thresh=0.4, bg_iou_thresh=0.3, box_loss="giou"),
}
RUN = ["retina_base", "retina_anchors", "retina_anchors_iou", "retina_anchors_iou_giou"]


def load_model(name):
    model = tv.build_model("retinanet", pretrained=False, **EXPERIMENTS[name])
    model.load_state_dict(torch.load(OUT / name / "best.pt", map_location="cpu"))
    return model.to(DEVICE).eval()


def run_experiment(name):
    model = tv.build_model("retinanet", pretrained=not DEBUG, **EXPERIMENTS[name])
    hist, _ = tv.train(model, P.prep, OUT / name, train_every=20 if DEBUG else 1, **PROTOCOL)
    model = load_model(name)
    dets = tv.predict(model, P.prep, "val", workers=PROTOCOL["workers"])
    c.save_json(dets, OUT / name / "val_dets.json")
    metrics = c.evaluate(GT, dets, per_class=True)
    ms = tv.measure_speed(model, P.prep, n=5 if DEBUG else 100)
    return an.log_result(P.results, name, "one-stage anchor-based", metrics, ms_per_img=ms, epochs=len(hist),
                         gpu_hours=sum(h["train_min"] for h in hist) / 60, config=str(EXPERIMENTS[name]))


for name in RUN:
    print("=" * 20, name)
    run_experiment(name)
    torch.cuda.empty_cache()

## 3. Результати і порівняння з two-stage

Рядки `two-stage` беруться з результатів `02_faster_rcnn`. На Kaggle додайте вихід того ноутбука як Input.

In [ ]:
COLS = ["family", "mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR_max", "ms_per_img", "gpu_hours"]
res = an.all_results(P)
res = res[res.family.isin(["one-stage anchor-based", "two-stage"])].set_index("name")
display(res[COLS].round(3))
display(res[[f"AP_{k}" for k in c.CLASSES]].round(3))
BEST = res.loc[[n for n in RUN if n in res.index], "mAP"].idxmax()
print("найкраща конфігурація RetinaNet:", BEST)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
for name in RUN:
    h = pd.DataFrame(c.load_json(OUT / name / "history.json"))
    axes[0].plot(h.epoch, h.classification, marker="o", label=name)
    axes[1].plot(h.epoch, h.mAP, marker="o", label=name)
    axes[2].plot(h.epoch, h.AP_small, marker="o", label=name)
for ax, t in zip(axes, ["focal loss (train)", "val mAP", "val AP_small"]):
    ax.set_title(t)
    ax.set_xlabel("епоха")
axes[1].legend(fontsize=8)
fig.savefig(P.images / "retina_curves.png", bbox_inches="tight", dpi=130)
plt.show()

## 4. Помилки найкращої RetinaNet

In [ ]:
dets = c.load_json(OUT / BEST / "val_dets.json")
d_df, g_df = an.error_report(GT, dets, P.data / "val", RAW, P.images, prefix="retina", score_thr=0.3)

## Висновки

<!-- Заповнюється після запуску: як anchors спираються на розміри об'єктів (розділ 1 і EDA); чи створює відсутність стадії пропозицій проблеми (дисбаланс фону й об'єктів, локалізація дрібних) і як їх знімають focal loss та пороги призначення; порівняння початкової і налаштованої RetinaNet з налаштованим Faster R-CNN (розділ 3). -->